# M4 · Conteo de pasajeros por segmentación de instancias (Bono +5)
**IAM-811 · Laboratorio Final — Sistema Inteligente de Control de Peaje**

Se cuentan las **personas visibles dentro del vehículo** (a través del parabrisas / ventanas) con
`yolo11s-seg` (clase `person`), y solo se cuentan las que **intersectan la máscara del vehículo** (M2).

> ### 🔒 Restricción de privacidad (obligatoria)
> * **Prohibido** el reconocimiento facial, la identificación de personas o el cálculo de *embeddings* de rostro.
> * **Nunca** se guardan recortes de personas ni de rostros (ni en disco, ni en Drive, ni en la BD).
> * Las visualizaciones pintan la persona con una **máscara opaca** y muestran solo el **número de pasajeros**.
> * El dataset de Roboflow se etiqueta **solo con polígonos de cuerpo** (`pasajero`), sin atributos de identidad.

**Métricas:** mAP50 / mAP50-95 (máscara) de la clase pasajero + **MAE del conteo** por vehículo.
**Salida:** `m4_pasajeros_seg.pt` (opcional: si no se entrena, el servidor usa `yolo11s-seg.pt` COCO).

## 1. Instalación

In [ ]:
!nvidia-smi -L || echo "ACTIVE LA GPU T4"
!pip -q install ultralytics==8.4.218 roboflow
import os, glob, json, shutil, yaml, cv2, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from ultralytics import YOLO
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

## 2. Funciones de conteo (idénticas a `inferencia/peaje_ia/m4_pasajeros.py`)

In [ ]:
VEHICULOS_COCO = {'car', 'bus', 'truck', 'motorcycle'}
PERSONA = {'person', 'persona', 'pasajero', 'passenger', 'conductor', 'driver'}
MIN_INTERSECCION = 0.30

def mascaras(r):
    out = []
    if r.masks is None: return out
    H, W = r.orig_shape
    for poly, c, s, b in zip(r.masks.xy, r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy(), r.boxes.xyxy.cpu().numpy()):
        m = np.zeros((H, W), np.uint8)
        if len(poly) >= 3: cv2.fillPoly(m, [poly.astype(np.int32)], 1)
        out.append({'nombre': r.names[c].lower(), 'conf': float(s), 'box': b, 'mask': m.astype(bool)})
    return out

def contenida(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    return max(0, x2 - x1) * max(0, y2 - y1) / max(1e-6, (a[2]-a[0]) * (a[3]-a[1]))

def contar_pasajeros(dets_personas, vehiculo):
    if vehiculo is None: return []
    dentro = []
    for p in dets_personas:
        frac = (p['mask'] & vehiculo['mask']).sum() / max(1, p['mask'].sum())
        if frac >= MIN_INTERSECCION and contenida(p['box'], vehiculo['box']) >= 0.6:
            dentro.append(p)
    return dentro

def dibujar_privado(img, vehiculo, pasajeros):
    # Máscara OPACA sobre cada pasajero: el rostro queda oculto. Solo se muestra el conteo.
    out = img.copy()
    if vehiculo is not None:
        x1, y1, x2, y2 = map(int, vehiculo['box']); cv2.rectangle(out, (x1, y1), (x2, y2), (219, 152, 52), 3)
    for p in pasajeros:
        out[p['mask']] = (0.15 * out[p['mask']] + 0.85 * np.array([90, 60, 30])).astype(np.uint8)
    cv2.putText(out, f'Pasajeros: {len(pasajeros)}', (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 4)
    return out

## 3. Dataset de Roboflow (Instance Segmentation)
Proyecto con 2 clases: **`pasajero`** (polígono del cuerpo visible dentro del vehículo) y **`vehiculo`**
(polígono de la carrocería). Imágenes frontales de minibuses, autos y buses con ocupantes visibles.
API key en **Colab → 🔑 Secretos → `ROBOFLOW_API_KEY`**.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
RF_WORKSPACE, RF_PROYECTO, RF_VERSION = 'tu-workspace', 'pasajeros-seg', 1     # <- cambie
ds = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY')).workspace(RF_WORKSPACE).project(RF_PROYECTO) \
        .version(RF_VERSION).download('yolov8')
DATA_YAML = f'{ds.location}/data.yaml'
cfg = yaml.safe_load(open(DATA_YAML)); print(cfg['names'])

## 4. Línea base: YOLO11s-seg COCO (modo *fallback* del servidor)

In [ ]:
def evaluar_conteo(modelo, nombres_persona=PERSONA, nombres_vehiculo=VEHICULOS_COCO | {'vehiculo'}):
    filas = []
    idx_pas = [i for i, n in enumerate(cfg['names']) if n.lower() in PERSONA]
    idx_veh = [i for i, n in enumerate(cfg['names']) if n.lower() == 'vehiculo']
    for ruta in sorted(glob.glob(f'{ds.location}/valid/images/*')):
        lbl = ruta.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        clases = [int(l.split()[0]) for l in open(lbl) if l.strip()] if os.path.exists(lbl) else []
        real = sum(c in idx_pas for c in clases)
        r = modelo.predict(ruta, conf=0.35, device=DEVICE, verbose=False)[0]
        dets = mascaras(r)
        veh = [d for d in dets if d['nombre'] in nombres_vehiculo]
        vehiculo = max(veh, key=lambda d: d['mask'].sum()) if veh else None
        pred = len(contar_pasajeros([d for d in dets if d['nombre'] in nombres_persona], vehiculo))
        filas.append({'imagen': os.path.basename(ruta), 'real': real, 'predicho': pred})
    df = pd.DataFrame(filas)
    return df, (df.real - df.predicho).abs().mean(), (df.real == df.predicho).mean() * 100

base = YOLO('yolo11s-seg.pt')
df_base, mae_base, acc_base = evaluar_conteo(base)
print(f'Línea base COCO -> MAE conteo = {mae_base:.3f} · exactitud = {acc_base:.1f}%')

## 5. Fine-tuning (pasajero + vehiculo)

In [ ]:
modelo = YOLO('yolo11s-seg.pt')
modelo.train(data=DATA_YAML, epochs=60, imgsz=640, batch=16, patience=20,
             project='/content/runs/segment', name='m4', exist_ok=True, device=DEVICE)
best = YOLO('/content/runs/segment/m4/weights/best.pt')
val = best.val(data=DATA_YAML, split='valid', project='/content/runs/segment', name='m4_val', exist_ok=True)
por_clase = pd.DataFrame({'clase': [best.names[i] for i in val.seg.ap_class_index],
                          'mAP50 mask': val.seg.ap50, 'mAP50-95 mask': val.seg.ap}).round(4)
display(por_clase)

## 6. Error de conteo del modelo ajustado

In [ ]:
df_ft, mae_ft, acc_ft = evaluar_conteo(best)
tabla = pd.DataFrame([
    {'modelo': 'yolo11s-seg COCO (fallback)', 'MAE conteo': mae_base, 'exactitud %': acc_base},
    {'modelo': 'm4_pasajeros_seg (fine-tuning)', 'MAE conteo': mae_ft, 'exactitud %': acc_ft,
     'mAP50 mask': val.seg.map50, 'mAP50-95 mask': val.seg.map}]).round(3)
json.dump(tabla.to_dict('records'), open('/content/m4_metricas.json', 'w'), indent=2)
tabla

## 7. Visualización con privacidad (máscara opaca + conteo, sin rostros)

In [ ]:
for ruta in sorted(glob.glob(f'{ds.location}/valid/images/*'))[:4]:
    img = cv2.imread(ruta)
    dets = mascaras(best.predict(img, conf=0.35, device=DEVICE, verbose=False)[0])
    veh = [d for d in dets if d['nombre'] not in PERSONA]
    v = max(veh, key=lambda d: d['mask'].sum()) if veh else None
    pas = contar_pasajeros([d for d in dets if d['nombre'] in PERSONA], v)
    plt.figure(figsize=(9, 6)); plt.imshow(dibujar_privado(img, v, pas)[:, :, ::-1]); plt.axis('off'); plt.show()
# Nota: no se escribe ninguna imagen de personas a disco.

## 8. Exportar `m4_pasajeros_seg.pt`

In [ ]:
shutil.copy('/content/runs/segment/m4/weights/best.pt', '/content/m4_pasajeros_seg.pt')
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/peaje-ia/pesos', exist_ok=True)
    shutil.copy('/content/m4_pasajeros_seg.pt', '/content/drive/MyDrive/peaje-ia/pesos/')
    shutil.copy('/content/m4_metricas.json', '/content/drive/MyDrive/peaje-ia/')
except Exception as e:
    print('Drive no montado:', e)
from google.colab import files
files.download('/content/m4_pasajeros_seg.pt'); files.download('/content/m4_metricas.json')